In [1]:
import numpy as np
from utils.config import load_config, Configuration

config = load_config("config.yaml")

In [2]:

subj = 1
mask_value = [7]


print(mask_value)

metadata = np.load(f"data/rdm_dir_stan_data/subj_{subj:02d}/subj_{subj:02d}/metadata.npy")


[7]


In [3]:
import os
from PIL import Image
#images = [Image.open(os.path.join(config.images_target_dir, f"{i}.jpg")) for i in metadata]
titles = [f"{i}" for i in metadata]


In [4]:

# Create and run the app
images = [os.path.join(config.directories.images_target_dir, f"{i}.jpg") for i in metadata]


In [7]:
def create_interactive_image_scatter_dash(images, image_pick: int, titles=None):
    """
    Create an interactive scatter plot using Dash where images are displayed at coordinates
    
    Parameters:
    data: numpy array of shape (n_samples, 2) containing x,y coordinates
    images: list of PIL Images or paths to image files
    titles: optional list of titles for each point
    """
    from dash import Dash, html, dcc
    import plotly.graph_objects as go
    from PIL import Image, ImageOps
    import base64
    import io
    import json
    import os
    import numpy as np
    from dash.dependencies import Input, Output

    subdir_path = os.path.join(
        config.directories.excel_files_target_dir, f"subj_{subj:02d}"
    )
    # Convert images to base64 strings and get their sizes
    with open(os.path.join(subdir_path, config.face_detection.results_path), "r") as f:
        face_detection_data = json.load(f)
    
    image_strings = []
    thumbnail_size = (200, 200)  # Size of images on the plot
    img_picked = images[image_pick]

    data_mds = []

    for mask_i, mask in enumerate(mask_value):
        #data/mds_dir_stan_data/subj_01/subj_01/mask_7_single_sample_0_both_mds.npy
        mds_path = f"data/mds_dir_stan_data/subj_{subj:02d}/subj_{subj:02d}/mask_7_single_sample_0_both_mds.npy"
        data = np.load(mds_path)

        data_mds.append([data[image_pick][0], data[image_pick][1]])
        
        file_basename = os.path.basename(img_picked)

        detection_bbox = [e for e in face_detection_data if e["file_name"] == file_basename][0]["detection"][0]
        if isinstance(img_picked, str):  # If it's a path
            img = Image.open(img_picked)
        img = img.crop((detection_bbox[0], detection_bbox[1], detection_bbox[2], detection_bbox[3]))

        if mask_i == len(mask_value) - 1:
            img = ImageOps.expand(img, border=10, fill="red")

        # Create thumbnail for plot
        img_thumb = img.copy()
        img_thumb.thumbnail(thumbnail_size)
        # Create full-size image for popup
        img_full = img.resize((400, 400))
        
        # Convert both versions to base64
        # Thumbnail
        thumb_buffered = io.BytesIO()
        img_thumb.save(thumb_buffered, format="PNG")
        thumb_str = base64.b64encode(thumb_buffered.getvalue()).decode()
        
        # Full size
        full_buffered = io.BytesIO()
        img_full.save(full_buffered, format="PNG")
        full_str = base64.b64encode(full_buffered.getvalue()).decode()
        
        image_strings.append({
            'thumbnail': thumb_str,
            'full': full_str
        })

    data_mds = np.array(data_mds)

    # Create Dash app
    app = Dash(__name__)

    # Create a figure with images
    fig = go.Figure()

    # Add invisible scatter points to maintain the plot range
    fig.add_trace(go.Scatter(
        x=[-1, 1],  # Force range by adding invisible points at the corners
        y=[-1, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))

    # Add the actual data points (also invisible)
    fig.add_trace(go.Scatter(
        x=data_mds[:, 0],
        y=data_mds[:, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))

    # Add images as layout shapes
    for i, (coords, img_data) in enumerate(zip(data_mds, image_strings)):
        title = titles[i] if titles else f"Point {i}"
        
        # Add image using layout images
        fig.add_layout_image(
            dict(
                source=f'data:image/png;base64,{img_data["thumbnail"]}',
                x=coords[0],
                y=coords[1],
                xref="x",
                yref="y",
                sizex=0.1,  # Reduced size to fit in [-1, 1] range
                sizey=0.1,  # Reduced size to fit in [-1, 1] range
                xanchor="center",
                yanchor="middle",
                layer="above"
            )
        )

    fig.update_layout(
        title="MDS Plot of Neural Response Similarity",
        xaxis_title="Dimension 1",
        yaxis_title="Dimension 2",
        hovermode='closest',
        # Set fixed range for both axes
        xaxis=dict(
            range=[-1, 1],
            fixedrange=True,  # Prevents zooming
            constrain='domain'  # Forces the range to be exactly [-1, 1]
        ),
        yaxis=dict(
            range=[-1, 1],
            fixedrange=True,  # Prevents zooming
            scaleanchor="x",
            scaleratio=1,
            constrain='domain'  # Forces the range to be exactly [-1, 1]
        ),
        # Ensure the plot maintains its range
        width=1500,
        height=1500,
        showlegend=False,
    )

    

    # Define the app layout
    app.layout = html.Div([
        dcc.Graph(
            id='scatter-plot',
            figure=fig,
            config={
                'doubleClick': 'reset',
                'scrollZoom': False,  # Disable scroll zoom
                'displayModeBar': True,  # Show the mode bar
                'modeBarButtonsToRemove': ['zoom', 'pan', 'select', 'lasso2d']  # Remove zoom and pan buttons
            }
        ),
        html.Div(id='image-container', style={'textAlign': 'center', 'marginTop': '20px'})
    ])

    # Callback to show larger image when clicking
    @app.callback(
        Output('image-container', 'children'),
        Input('scatter-plot', 'clickData')
    )
    def display_image(clickData):
        if clickData is None:
            return html.Div("Click an image to view it larger")
        
        point_index = clickData['points'][0]['pointIndex']
        img_src = f'data:image/png;base64,{image_strings[point_index]["full"]}'
        title = titles[point_index] if titles else f"Point {point_index}"
        
        return html.Div([
            html.H4(title),
            html.Img(src=img_src, style={'maxWidth': '400px'})
        ])
    
    save_path = os.path.join("data", "interesting_stuff", f"mds_sample_{image_pick}.png")
    fig.write_image(save_path, scale=2)

    return app

In [8]:

for i in range(1):
    app = create_interactive_image_scatter_dash(images, i)


app.run_server(debug=True)

In [9]:
def create_interactive_image_scatter_dash(images, titles=None):
    """
    Create an interactive Dash app that shows a scatter plot of images positioned
    by MDS coordinates. Each point is a thumbnail of a cropped face under different mask conditions.
    
    Parameters:
    - images: list of image paths or PIL images
    - titles: optional list of titles for each image
    """
    from dash import Dash, html, dcc
    import plotly.graph_objects as go
    from PIL import Image, ImageOps
    import base64
    import io
    import json
    import os
    import numpy as np
    from dash.dependencies import Input, Output

    image_strings = []
    coords_list = []
    labels = []

    thumbnail_size = (200, 200)

    # Load face detection info
    subdir_path = os.path.join(config.directories.excel_files_target_dir, f"subj_{subj:02d}")
    with open(os.path.join(subdir_path, config.face_detection.results_path), "r") as f:
        face_detection_data = json.load(f)

    for image_idx, img_path in enumerate(images):
        file_basename = os.path.basename(img_path)

        detection = [e for e in face_detection_data if e["file_name"] == file_basename]
        if not detection:
            continue  # skip if no face detection info
        
        if len(detection[0]["detection"]) == 0:
            continue
        detection_bbox = detection[0]["detection"][0]

        for mask_i, mask in enumerate(mask_value):
            # Load MDS coords
            mds_path = f"data/mds_dir_stan_data/subj_{subj:02d}/subj_{subj:02d}/mask_7_single_sample_0_both_mds.npy"
            # mds_path = f"data/mds_dir_stan_data/subj_{subj:02d}/subj_{subj:02d}/mask_{mask}_single_sample_0_both_mds.npy"
            data = np.load(mds_path)

            coords = data[image_idx][:2]
            coords_list.append(coords)
            labels.append(f"{titles[image_idx]} (mask {mask})" if titles else f"Image {image_idx} - mask {mask}")

            img = Image.open(img_path)
            img = img.crop((detection_bbox[0], detection_bbox[1], detection_bbox[2], detection_bbox[3]))

            #if mask_i == len(mask_value) - 1:
            #    img = ImageOps.expand(img, border=10, fill="red")

            # Thumbnail
            img_thumb = img.copy()
            img_thumb.thumbnail(thumbnail_size)
            thumb_buffered = io.BytesIO()
            img_thumb.save(thumb_buffered, format="PNG")
            thumb_str = base64.b64encode(thumb_buffered.getvalue()).decode()

            # Full size
            img_full = img.resize((400, 400))
            full_buffered = io.BytesIO()
            img_full.save(full_buffered, format="PNG")
            full_str = base64.b64encode(full_buffered.getvalue()).decode()

            image_strings.append({
                'thumbnail': thumb_str,
                'full': full_str
            })

    coords_array = np.array(coords_list)

    # Create Dash app
    app = Dash(__name__)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=[-1, 1],
        y=[-1, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter(
        x=coords_array[:, 0],
        y=coords_array[:, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))

    for i, (coords, img_data) in enumerate(zip(coords_array, image_strings)):
        fig.add_layout_image(dict(
            source=f'data:image/png;base64,{img_data["thumbnail"]}',
            x=coords[0],
            y=coords[1],
            xref="x",
            yref="y",
            sizex=0.1,
            sizey=0.1,
            xanchor="center",
            yanchor="middle",
            layer="above"
        ))

    fig.update_layout(
        title="MDS Plot of Neural Response Similarity (All Images)",
        xaxis=dict(range=[-1, 1], fixedrange=True, constrain='domain'),
        yaxis=dict(range=[-1, 1], fixedrange=True, scaleanchor='x', scaleratio=1, constrain='domain'),
        hovermode='closest',
        width=1500,
        height=1500,
        showlegend=False
    )

    app.layout = html.Div([
        dcc.Graph(
            id='scatter-plot',
            figure=fig,
            config={
                'doubleClick': 'reset',
                'scrollZoom': False,
                'displayModeBar': True,
                'modeBarButtonsToRemove': ['zoom', 'pan', 'select', 'lasso2d']
            }
        ),
        html.Div(id='image-container', style={'textAlign': 'center', 'marginTop': '20px'})
    ])

    @app.callback(
        Output('image-container', 'children'),
        Input('scatter-plot', 'clickData')
    )
    def display_image(clickData):
        if clickData is None:
            return html.Div("Click an image to view it larger")

        point_index = clickData['points'][0]['pointIndex']
        img_src = f'data:image/png;base64,{image_strings[point_index]["full"]}'
        title = labels[point_index]

        return html.Div([
            html.H4(title),
            html.Img(src=img_src, style={'maxWidth': '400px'})
        ])

    return app


In [10]:
app = create_interactive_image_scatter_dash(images)

app.run_server(debug=True)

In [13]:
def create_interactive_image_scatter_dash(images, titles=None):
    """
    Create an interactive Dash app that shows a scatter plot of images positioned
    by MDS coordinates. Each point shows an image that is generated as follows:
      - A black image of the same size as the original is created.
      - A filled white rectangle is drawn on the black image at the location defined 
        by the face detection bounding box.
      - This composite image is then resized to a thumbnail for the scatter plot, 
        and a larger version is generated for display on click.
    
    Parameters:
    - images: list of image paths or PIL images
    - titles: optional list of titles for each image
    """
    from dash import Dash, html, dcc
    import plotly.graph_objects as go
    from PIL import Image, ImageDraw
    import base64
    import io
    import json
    import os
    import numpy as np
    from dash.dependencies import Input, Output

    image_strings = []
    coords_list = []
    labels = []

    thumbnail_size = (200, 200)

    # Load face detection info
    subdir_path = os.path.join(config.directories.excel_files_target_dir, f"subj_{subj:02d}")
    with open(os.path.join(subdir_path, config.face_detection.results_path), "r") as f:
        face_detection_data = json.load(f)

    for image_idx, img_path in enumerate(images):
        file_basename = os.path.basename(img_path)

        # Find face detection data for the image
        detection = [e for e in face_detection_data if e["file_name"] == file_basename]
        if not detection:
            continue  # skip if no face detection info
        
        if len(detection[0]["detection"]) == 0:
            continue
        detection_bbox = detection[0]["detection"][0]  # (left, top, right, bottom)

        for mask_i, mask in enumerate(mask_value):
            # Load MDS coords
            mds_path = f"data/mds_dir_stan_data/subj_{subj:02d}/subj_{subj:02d}/mask_7_single_sample_0_both_mds.npy"
            data = np.load(mds_path)

            coords = data[image_idx][:2]
            coords_list.append(coords)
            labels.append(f"{titles[image_idx]} (mask {mask})" if titles else f"Image {image_idx} - mask {mask}")

            # --- Updated Image Processing ---
            # Open the original image to get its size
            img = Image.open(img_path).convert("RGB")
            
            # Create a black image with the same size as the original image
            black_img = Image.new("RGB", img.size, color="black")
            
            # Draw a filled white rectangle on the black image where the face bbox is located
            draw = ImageDraw.Draw(black_img)
            bbox = (detection_bbox[0], detection_bbox[1], detection_bbox[2], detection_bbox[3])
            draw.rectangle(bbox, fill="white")
            # --------------------------------

            # Create thumbnail version (resize to predefined thumbnail_size)
            img_thumb = black_img.copy().resize(thumbnail_size)
            thumb_buffered = io.BytesIO()
            img_thumb.save(thumb_buffered, format="PNG")
            thumb_str = base64.b64encode(thumb_buffered.getvalue()).decode()

            # Create full-sized version (resize to 400x400)
            img_full = black_img.copy().resize((400, 400))
            full_buffered = io.BytesIO()
            img_full.save(full_buffered, format="PNG")
            full_str = base64.b64encode(full_buffered.getvalue()).decode()

            image_strings.append({
                'thumbnail': thumb_str,
                'full': full_str
            })

    coords_array = np.array(coords_list)

    # Create Dash app
    app = Dash(__name__)

    fig = go.Figure()

    # Add invisible scatter points to force the axis range
    fig.add_trace(go.Scatter(
        x=[-1, 1],
        y=[-1, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter(
        x=coords_array[:, 0],
        y=coords_array[:, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))

    # Add each image as an overlay on the scatter plot
    for i, (coords, img_data) in enumerate(zip(coords_array, image_strings)):
        fig.add_layout_image(dict(
            source=f'data:image/png;base64,{img_data["thumbnail"]}',
            x=coords[0],
            y=coords[1],
            xref="x",
            yref="y",
            sizex=0.1,
            sizey=0.1,
            xanchor="center",
            yanchor="middle",
            layer="above"
        ))

    fig.update_layout(
        title="MDS Plot of Neural Response Similarity (All Images)",
        xaxis=dict(range=[-1, 1], fixedrange=True, constrain='domain'),
        yaxis=dict(range=[-1, 1], fixedrange=True, scaleanchor='x', scaleratio=1, constrain='domain'),
        hovermode='closest',
        width=1500,
        height=1500,
        showlegend=False
    )

    app.layout = html.Div([
        dcc.Graph(
            id='scatter-plot',
            figure=fig,
            config={
                'doubleClick': 'reset',
                'scrollZoom': False,
                'displayModeBar': True,
                'modeBarButtonsToRemove': ['zoom', 'pan', 'select', 'lasso2d']
            }
        ),
        html.Div(id='image-container', style={'textAlign': 'center', 'marginTop': '20px'})
    ])

    @app.callback(
        Output('image-container', 'children'),
        Input('scatter-plot', 'clickData')
    )
    def display_image(clickData):
        if clickData is None:
            return html.Div("Click an image to view it larger")

        point_index = clickData['points'][0]['pointIndex']
        img_src = f'data:image/png;base64,{image_strings[point_index]["full"]}'
        title = labels[point_index]

        return html.Div([
            html.H4(title),
            html.Img(src=img_src, style={'maxWidth': '400px'})
        ])

    return app


In [14]:
app = create_interactive_image_scatter_dash(images)

app.run_server(debug=True)